In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') # Suppress annoying convergence warnings for LR/SVM

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb

print("[*] INITIATING THE MULTI-MODEL BENCHMARK SHOWDOWN...")

# ---------------------------------------------------------
# 1. LOAD AND PREPARE THE EUROPEAN DATA
# ---------------------------------------------------------
processed_dir = "/workspace/data/processed"
clinical_df = pd.read_csv(os.path.join(processed_dir, "GSE65682_clinical_clean.csv"), index_col=0)
expr_df = pd.read_csv(os.path.join(processed_dir, "GSE65682_expression_TOP100.csv"), index_col=0)

full_df = clinical_df.join(expr_df, how='inner')

# Encode clinical targets
full_df['gender_num'] = full_df['gender'].apply(lambda x: 1 if x == 'male' else 0)
full_df['pneumonia'] = full_df['pneumonia diagnoses'].apply(lambda x: 1 if x == 'cap' else 0)
full_df['diabetes'] = full_df['diabetes_mellitus'].apply(lambda x: 1 if x == 'yes' else 0)

# Define Features and Target
all_features = ['age', 'gender_num', 'pneumonia', 'diabetes'] + expr_df.columns.tolist()
X_raw = full_df[all_features].values
y = full_df['mortality_event_28days'].values

# Standardize the data (Critical for SVM, KNN, and Logistic Regression)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# ---------------------------------------------------------
# 2. INITIALIZE THE 7 CONTENDERS
# ---------------------------------------------------------
imbalance_ratio = sum(y==0) / sum(y==1)

models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "Support Vector Machine": SVC(probability=True, class_weight='balanced', random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=100, scale_pos_weight=imbalance_ratio, eval_metric='logloss', random_state=42)
}

# ---------------------------------------------------------
# 3. RUN THE BRUTAL 5-FOLD CROSS VALIDATION
# ---------------------------------------------------------
print("[*] Running 5-Fold Cross-Validation across 7 distinct architectures...\n")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['roc_auc', 'f1', 'precision', 'recall']

results_list = []

for name, model in models.items():
    # Run the rigorous cross-validation
    cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring_metrics, n_jobs=-1)
    
    # Calculate the mean scores across the 5 folds
    mean_auc = np.mean(cv_results['test_roc_auc'])
    mean_f1 = np.mean(cv_results['test_f1'])
    mean_precision = np.mean(cv_results['test_precision'])
    mean_recall = np.mean(cv_results['test_recall'])
    
    results_list.append({
        "Model": name,
        "AUROC": mean_auc,
        "F1 Score": mean_f1,
        "Precision": mean_precision,
        "Recall": mean_recall
    })

# ---------------------------------------------------------
# 4. PRINT THE PUBLICATION-READY LEADERBOARD
# ---------------------------------------------------------
results_df = pd.DataFrame(results_list)
# Sort by AUROC descending to see who won
results_df = results_df.sort_values(by="AUROC", ascending=False).reset_index(drop=True)

print("="*75)
print(f"{'FINAL ALGORITHM LEADERBOARD (European Cohort Internal Validation)':^75}")
print("="*75)
# Format the dataframe to display 4 decimal places just like a research paper
print(results_df.to_string(formatters={
    'AUROC': '{:,.4f}'.format,
    'F1 Score': '{:,.4f}'.format,
    'Precision': '{:,.4f}'.format,
    'Recall': '{:,.4f}'.format
}))
print("="*75)

# Save the table to disk for your paper
results_df.to_csv(os.path.join(processed_dir, "benchmark_model_comparison.csv"), index=False)
print("\n[+] Saved complete leaderboard to 'benchmark_model_comparison.csv'")

[*] INITIATING THE MULTI-MODEL BENCHMARK SHOWDOWN...
[*] Running 5-Fold Cross-Validation across 7 distinct architectures...

     FINAL ALGORITHM LEADERBOARD (European Cohort Internal Validation)     
                    Model  AUROC F1 Score Precision Recall
0                 XGBoost 0.8782   0.6085    0.7701 0.5087
1           Random Forest 0.8022   0.1821    0.6933 0.1055
2  Support Vector Machine 0.7891   0.5320    0.5565 0.5186
3     Logistic Regression 0.7475   0.5182    0.4558 0.6051
4             Naive Bayes 0.7397   0.5061    0.4845 0.5356
5     K-Nearest Neighbors 0.6693   0.3042    0.5103 0.2190
6           Decision Tree 0.6393   0.4430    0.5312 0.3854

[+] Saved complete leaderboard to 'benchmark_model_comparison.csv'


In [2]:
import numpy as np
import torch
from sklearn.metrics import roc_auc_score

print("[*] INITIATING REAL-WORLD STRESS TESTS...")

# Assuming model, Xc_test_tensor, Xg_test_tensor, and y_test are already in your memory
# from the Australian Validation script.

def test_robustness(model, clin_tensor, gen_tensor, y_true, missing_rate=0.0, noise_std=0.0):
    model.eval()
    
    # 1. Simulate Missing Data (Zeroing out random genes)
    if missing_rate > 0:
        mask = np.random.rand(*gen_tensor.shape) > missing_rate
        gen_tensor_mutilated = gen_tensor * torch.FloatTensor(mask)
    else:
        gen_tensor_mutilated = gen_tensor.clone()
        
    # 2. Simulate Hardware Noise
    if noise_std > 0:
        noise = torch.randn(*gen_tensor_mutilated.shape) * noise_std
        gen_tensor_mutilated += noise

    with torch.no_grad():
        predictions = model(clin_tensor, gen_tensor_mutilated)
        probs = torch.sigmoid(predictions).numpy()
        auroc = roc_auc_score(y_true, probs)
        
    return auroc

print("\n--- STRESS TEST 1: LAB FAILURES (Missing Genes) ---")
rates = [0.0, 0.10, 0.30, 0.50, 0.75]
for rate in rates:
    score = test_robustness(model, Xc_test_tensor, Xg_test_tensor, y_test, missing_rate=rate)
    print(f"Missing {int(rate*100)}% of Genes | AUROC: {score:.4f}")

print("\n--- STRESS TEST 2: BAD HARDWARE (Gaussian Noise) ---")
noise_levels = [0.1, 0.5, 1.0, 2.0]
for noise in noise_levels:
    score = test_robustness(model, Xc_test_tensor, Xg_test_tensor, y_test, noise_std=noise)
    print(f"Noise Variance (+/- {noise}) | AUROC: {score:.4f}")

[*] INITIATING REAL-WORLD STRESS TESTS...

--- STRESS TEST 1: LAB FAILURES (Missing Genes) ---


NameError: name 'Xc_test_tensor' is not defined